# Indian Traffic Sign Recognition with a CNN

**Goal:** classify images of Indian traffic signs into **58 classes** (speed limits, no entry, turns, junctions,
level crossings, parking and more) with a convolutional neural network (CNN) built in TensorFlow/Keras.

**Data:** about 14,000 labelled traffic sign images, one folder per class, plus `traffic_sign.csv`
which maps each class ID to the sign's name.

**Workflow:** load and inspect the data, train a CNN, then evaluate it properly with a classification report,
confusion analysis and example predictions.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

tf.random.set_seed(42)
np.random.seed(42)
print("TensorFlow", tf.__version__)

## 1. Load the data

Place the dataset like this (see the README for the download link):

```
data/
├── traffic_sign.csv      # ClassId, Name
└── Images/
    ├── 0/   ...images of class 0
    ├── 1/
    └── ...  (58 folders)
```

In [ ]:
CSV_PATH = "data/traffic_sign.csv"
IMAGE_DIR = "data/Images"

labels = pd.read_csv(CSV_PATH)
class_names = dict(zip(labels["ClassId"], labels["Name"]))
print(len(class_names), "classes")
labels.head()

In [ ]:
# Number of images per class
counts = pd.Series({int(c): len(os.listdir(os.path.join(IMAGE_DIR, c)))
                    for c in os.listdir(IMAGE_DIR)}).sort_index()
print(f"{counts.sum():,} images in total, {counts.min()}-{counts.max()} per class")

counts.plot(kind="bar", figsize=(14, 3), color="steelblue")
plt.title("Images per class"); plt.xlabel("Class ID"); plt.ylabel("Images")
plt.tight_layout(); plt.savefig("images/class_distribution.png", dpi=120); plt.show()

## 2. Prepare training and validation data

Images are resized to 64x64 pixels and pixel values scaled to 0-1. 20% of the images are kept aside for validation.

**Important:** the validation generator uses `shuffle=False`. If it were shuffled, the order of the predictions
would not match the order of the true labels (`validation_generator.classes`), and the evaluation would be wrong.

In [ ]:
IMG_SIZE = (64, 64)
BATCH_SIZE = 32
EPOCHS = 20

datagen = ImageDataGenerator(rescale=1.0 / 255, validation_split=0.2)

train_generator = datagen.flow_from_directory(
    IMAGE_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", subset="training", shuffle=True, seed=42)

validation_generator = datagen.flow_from_directory(
    IMAGE_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", subset="validation", shuffle=False)

# Keras sorts folder names as text ("0", "1", "10", "11", ...), so the model's output index is not the
# same as the ClassId. Build the correct name for each output index from class_indices.
index_to_name = {idx: class_names[int(folder)]
                 for folder, idx in train_generator.class_indices.items()}

In [ ]:
# Look at some training images
images_batch, labels_batch = next(train_generator)
plt.figure(figsize=(12, 5))
for i in range(12):
    plt.subplot(2, 6, i + 1)
    plt.imshow(images_batch[i])
    plt.title(index_to_name[labels_batch[i].argmax()], fontsize=8)
    plt.axis("off")
plt.tight_layout(); plt.savefig("images/sample_images.png", dpi=120); plt.show()

## 3. Build and train the CNN

In [ ]:
model = Sequential([
    Input(shape=(*IMG_SIZE, 3)),
    Conv2D(32, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5),                                   # reduces overfitting
    Dense(train_generator.num_classes, activation="softmax"),
])
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
history = model.fit(train_generator, validation_data=validation_generator, epochs=EPOCHS)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ["accuracy", "loss"]):
    ax.plot(history.history[metric], label="Training")
    ax.plot(history.history[f"val_{metric}"], label="Validation")
    ax.set_title(f"Model {metric}"); ax.set_xlabel("Epoch"); ax.legend()
plt.tight_layout(); plt.savefig("images/training_curves.png", dpi=120); plt.show()

## 4. Evaluate the model

In [ ]:
val_loss, val_accuracy = model.evaluate(validation_generator, verbose=0)
print(f"Validation accuracy: {val_accuracy:.2%}")

y_prob = model.predict(validation_generator, verbose=0)
y_pred = y_prob.argmax(axis=1)
y_true = validation_generator.classes

# Sanity check: this must match the accuracy above
print(f"Accuracy from predictions: {(y_pred == y_true).mean():.2%}")

In [ ]:
target_names = [f"{index_to_name[i]} ({i})" for i in range(len(index_to_name))]
report = classification_report(y_true, y_pred, target_names=target_names, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report).T.iloc[:-3].sort_values("f1-score")
print(classification_report(y_true, y_pred, target_names=target_names, zero_division=0))

In [ ]:
# The 10 classes the model finds hardest
report_df["f1-score"].head(10).plot(kind="barh", color="indianred", figsize=(8, 4))
plt.title("10 hardest classes (lowest F1-score)"); plt.xlabel("F1-score")
plt.tight_layout(); plt.savefig("images/hardest_classes.png", dpi=120); plt.show()

In [ ]:
# Which pairs of signs does the model confuse most often?
cm = confusion_matrix(y_true, y_pred)
np.fill_diagonal(cm, 0)
pairs = [(cm[i, j], index_to_name[i], index_to_name[j])
         for i in range(len(cm)) for j in range(len(cm)) if cm[i, j] > 0]
pd.DataFrame(sorted(pairs, reverse=True)[:10], columns=["Count", "True sign", "Predicted as"])

In [ ]:
# Example predictions on validation images (green = correct, red = wrong)
plt.figure(figsize=(14, 6))
for k, i in enumerate(np.random.choice(len(y_true), 12, replace=False)):
    img = plt.imread(validation_generator.filepaths[i])
    plt.subplot(2, 6, k + 1)
    plt.imshow(img)
    ok = y_pred[i] == y_true[i]
    plt.title(f"{index_to_name[y_pred[i]]}\n({y_prob[i].max():.0%})", fontsize=8,
              color="green" if ok else "red")
    plt.axis("off")
plt.tight_layout(); plt.savefig("images/example_predictions.png", dpi=120); plt.show()

In [ ]:
import json
os.makedirs("models", exist_ok=True)
model.save("models/traffic_sign_cnn.keras")
with open("models/class_names.json", "w") as f:          # used by the Streamlit app
    json.dump({int(k): v for k, v in index_to_name.items()}, f, indent=1)
print("Model and class names saved to the models/ folder")

## 5. Conclusions

- A compact CNN with two convolution blocks learns to recognise 58 Indian traffic sign classes from 64x64 images.
- Validation accuracy and the hardest classes are shown above. Most errors are between signs that look alike,
  such as the different junction and level-crossing signs.
- Training accuracy ends above validation accuracy, which shows some overfitting.

**Possible next steps:** data augmentation (small rotations, shifts and zoom), batch normalisation, early stopping,
and transfer learning with a pre-trained model such as MobileNetV2.